# Package Imports


In [1]:
import gymnasium as gym
import gym_pusht
from rich import print

In [2]:
import torch
import math
import numpy as np

In [3]:
import collections

In [4]:
import lerobot
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

/home/thankgod/2025/msc_project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler
from tqdm.auto import tqdm

In [6]:
import sys
from pathlib import Path

data_path = Path.cwd().parent / "data"
env_path = Path.cwd().parent / "env"

sys.path.insert(0, str(data_path))
sys.path.insert(1, str(env_path))

In [7]:
# Now import
from image_dataset import PushTImageDataset, normalize_data, unnormalize_data
from pusht_image_env import PushTImageEnv

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/home/thankgod/2025/msc_project/.venv/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


# Local Environment Setup


In [8]:

# 0. create env object
env = PushTImageEnv()

# 1. seed env for initial state.
# Seed 0-200 are used for the demonstration dataset.
env.seed(1000)

# 2. must reset before use
obs, info = env.reset()

# 3. 2D positional action space [0,512]
action = env.action_space.sample()

# 4. Standard gym step method
obs, reward, terminated, truncated, info = env.step(action)

# prints and explains each dimension of the observation and action vectors
with np.printoptions(precision=4, suppress=True, threshold=5):
    print("obs['image'].shape:", obs['image'].shape, "float32, [0,1]")
    print("obs['agent_pos'].shape:", obs['agent_pos'].shape, "float32, [0,512]")
    print("action.shape: ", action.shape, "float32, [0,512]")

obs['image'].shape:
(3, 96, 96)
float32, [0,1]

obs['agent_pos'].shape:
(2,)
float32, [0,512]

action.shape: 
(2,)
float32, [0,512]

# Environment SetUp


In [9]:
pusht_pixels_env = gym.make(
    "gym_pusht/PushT-v0",
    obs_type="pixels",
    render_mode="rgb_array",
)
pusht_pixels_env.reset()

action = pusht_pixels_env.action_space.sample()
observation, reward, terminated, truncated, info = pusht_pixels_env.step(action)

print("Observation:", observation)
print("Observation:", observation.shape)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print("Info:", info)

Observation: [[[255 255 255]
  [248 248 248]
  [248 248 248]
  ...
  [248 248 248]
  [248 248 248]
  [255 255 255]]

 [[248 248 248]
  [222 222 222]
  [233 233 233]
  ...
  [233 233 233]
  [222 222 222]
  [248 248 248]]

 [[247 247 247]
  [233 233 233]
  [255 255 255]
  ...
  [255 255 255]
  [233 233 233]
  [247 247 247]]

 ...

 [[247 247 247]
  [233 233 233]
  [255 255 255]
  ...
  [255 255 255]
  [233 233 233]
  [247 247 247]]

 [[248 248 248]
  [222 222 222]
  [233 233 233]
  ...
  [233 233 233]
  [222 222 222]
  [248 248 248]]

 [[255 255 255]
  [248 248 248]
  [248 248 248]
  ...
  [248 248 248]
  [248 248 248]
  [255 255 255]]]

Observation:
(96, 96, 3)

Reward: 0.006253023575694731

Terminated: False

Truncated: False

Info:
{
    'pos_agent': array([246.63502823, 165.14495896]),
    'vel_agent': array([-1219.41727312,   685.15869401]),
    'block_pose': array([148.37073655, 385.09895253,  -0.84401499]),
    'goal_pose': array([256.        , 256.        ,   0.78539816]),
    'n_contacts': 0,
    'is_success': False,
    'coverage': 0.005940372396909995
}

In [10]:
pusht_pixels_env = gym.make(
    "gym_pusht/PushT-v0",
    obs_type="pixels",
    render_mode="human",
)
pusht_pixels_env.reset()

for _ in range(200):
    action = pusht_pixels_env.action_space.sample()  # random action policy
    observation, reward, terminated, truncated, info = pusht_pixels_env.step(action)
    image = pusht_pixels_env.render()

    if terminated or truncated:
        observation, info = pusht_pixels_env.reset()

pusht_pixels_env.close()

/home/thankgod/2025/msc_project/.venv/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:131: UserWarning: WARN: The obs returned by the `reset()` method was expecting a numpy array, actual type: <class 'NoneType'>
  logger.warn(
/home/thankgod/2025/msc_project/.venv/lib/python3.10/site-packages/gymnasium/spaces/box.py:240: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")
/home/thankgod/2025/msc_project/.venv/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:159: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/home/thankgod/2025/msc_project/.venv/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:131: UserWarning: WARN: The obs returned by the `step()` method was expecting a numpy array, actual type: <class 'NoneType'>
  logger.warn(
/home/thankgod/2025/msc_project/.venv/

# Dataset


In [11]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("lerobot/pusht_image")

ValueError: Feature type 'List' not found. Available feature types: ['Value', 'ClassLabel', 'Translation', 'TranslationVariableLanguages', 'LargeList', 'Sequence', 'Array2D', 'Array3D', 'Array4D', 'Array5D', 'Audio', 'Image', 'Video', 'Pdf', 'VideoFrame']

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['observation.image', 'observation.state', 'action', 'episode_index', 'frame_index', 'timestamp', 'next.reward', 'next.done', 'next.success', 'index', 'task_index'],
        num_rows: 48336
    })
})

In [ ]:
ds["train"]["observation.image"]

Column([<PIL.PngImagePlugin.PngImageFile image mode=RGB size=96x96 at 0x7410F00F0E20>, <PIL.PngImagePlugin.PngImageFile image mode=RGB size=96x96 at 0x7410F00F1D50>, <PIL.PngImagePlugin.PngImageFile image mode=RGB size=96x96 at 0x7410F00F16C0>, <PIL.PngImagePlugin.PngImageFile image mode=RGB size=96x96 at 0x7410F00F1630>, <PIL.PngImagePlugin.PngImageFile image mode=RGB size=96x96 at 0x7410F00F1600>])

In [ ]:
ds["train"]["observation.state"]

Column([[222.0, 97.0], [225.2523956298828, 89.31253051757812], [227.5923309326172, 84.53437805175781], [228.420166015625, 84.27986145019531], [229.04222106933594, 84.95709991455078]])

In [ ]:
ds["train"]["action"]

Column([[233.0, 71.0], [229.0, 83.0], [229.0, 86.0], [230.0, 86.0], [239.0, 89.0]])

In [12]:
dataset_image = LeRobotDataset("lerobot/pusht_image")
dataset_image

LeRobotDataset({
    Repository ID: 'lerobot/pusht_image',
    Number of selected episodes: '206',
    Number of selected samples: '25650',
    Features: '['observation.image', 'observation.state', 'action', 'episode_index', 'frame_index', 'timestamp', 'next.reward', 'next.done', 'next.success', 'index', 'task_index']',
})',

In [13]:
dataset_image

LeRobotDataset({
    Repository ID: 'lerobot/pusht_image',
    Number of selected episodes: '206',
    Number of selected samples: '25650',
    Features: '['observation.image', 'observation.state', 'action', 'episode_index', 'frame_index', 'timestamp', 'next.reward', 'next.done', 'next.success', 'index', 'task_index']',
})',

In [14]:
dataset_image[0]

{'observation.image': tensor([[[1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000],
          [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
          [0.9686, 0.9137, 1.0000,  ..., 1.0000, 0.9137, 0.9686],
          ...,
          [0.9686, 0.9137, 1.0000,  ..., 1.0000, 0.9137, 0.9686],
          [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
          [1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000]],
 
         [[1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000],
          [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
          [0.9686, 0.9137, 1.0000,  ..., 1.0000, 0.9137, 0.9686],
          ...,
          [0.9686, 0.9137, 1.0000,  ..., 1.0000, 0.9137, 0.9686],
          [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
          [1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000]],
 
         [[1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000],
          [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
          [0.9686, 

In [15]:
for i in range(10):
    print(dataset_image[i]["observation.state"])

tensor([222.,  97.])

tensor([225.2524,  89.3125])

tensor([227.5923,  84.5344])

tensor([228.4202,  84.2799])

tensor([229.0422,  84.9571])

tensor([232.1624,  86.3440])

tensor([238.8461,  89.3548])

tensor([248.0901,  94.0601])

tensor([258.1464,  99.5915])

tensor([268.2669, 105.9949])

In [16]:
delta_timestamps = {
    # loads 4 images: 1 second before current frame, 500 ms before, 200 ms before, and current frame
    "observation.image": [-1, -0.5, -0.20, 0],
    # loads 6 state vectors: 1.5 seconds before, 1 second before, ... 200 ms, 100 ms, and current frame
    "observation.state": [-1.5, -1, -0.5, -0.20, -0.10, 0],
    # loads 16 action vectors: current frame, 1 frame in the future, 2 frames, ... 16 frames in the future
    "action": [t / dataset_image.fps for t in range(16)],
}
# Note that in any case, these delta_timestamps values need to be multiples of (1/fps) so that added to any
# timestamp, you still get a valid timestamp.

dataset = LeRobotDataset(
    "lerobot/pusht_image",
    delta_timestamps=delta_timestamps,
)

In [17]:
print(dataset[0]["observation.image"].shape)  # (4, c, h, w) -> (4, 3, 96, 96)
print(dataset[0]["observation.state"].shape)  # (6, c) -> (6, 2) -> (To, Do)
print(dataset[0]["action"].shape)  # (16, c) -> (16, 2) -> (Ta, Da)

torch.Size([4, 3, 96, 96])

torch.Size([6, 2])

torch.Size([16, 2])

## Dataset Used


In [18]:
delta_timestamps = {
    "observation.image": [-0.1, 0.0],
    "observation.state": [-0.1, 0.0],
    "action": [
        -0.1,
        0.0,
        0.1,
        0.2,
        0.3,
        0.4,
        0.5,
        0.6,
        0.7,
        0.8,
        0.9,
        1.0,
        1.1,
        1.2,
        1.3,
        1.4,
    ],
}
dataset = LeRobotDataset(
    "lerobot/pusht_image",
    delta_timestamps=delta_timestamps,
)
dataset

LeRobotDataset({
    Repository ID: 'lerobot/pusht_image',
    Number of selected episodes: '206',
    Number of selected samples: '25650',
    Features: '['observation.image', 'observation.state', 'action', 'episode_index', 'frame_index', 'timestamp', 'next.reward', 'next.done', 'next.success', 'index', 'task_index']',
})',

In [19]:
dataset.meta.camera_keys

['observation.image']

In [20]:
dataset[0]

{'observation.image': tensor([[[[1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000],
           [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
           [0.9686, 0.9137, 1.0000,  ..., 1.0000, 0.9137, 0.9686],
           ...,
           [0.9686, 0.9137, 1.0000,  ..., 1.0000, 0.9137, 0.9686],
           [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
           [1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000]],
 
          [[1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000],
           [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
           [0.9686, 0.9137, 1.0000,  ..., 1.0000, 0.9137, 0.9686],
           ...,
           [0.9686, 0.9137, 1.0000,  ..., 1.0000, 0.9137, 0.9686],
           [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
           [1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000]],
 
          [[1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000],
           [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
   

In [21]:
for i in range(10):
    print(dataset[i]["observation.state"])

tensor([[222.,  97.],
        [222.,  97.]])

tensor([[222.0000,  97.0000],
        [225.2524,  89.3125]])

tensor([[225.2524,  89.3125],
        [227.5923,  84.5344]])

tensor([[227.5923,  84.5344],
        [228.4202,  84.2799]])

tensor([[228.4202,  84.2799],
        [229.0422,  84.9571]])

tensor([[229.0422,  84.9571],
        [232.1624,  86.3440]])

tensor([[232.1624,  86.3440],
        [238.8461,  89.3548]])

tensor([[238.8461,  89.3548],
        [248.0901,  94.0601]])

tensor([[248.0901,  94.0601],
        [258.1464,  99.5915]])

tensor([[258.1464,  99.5915],
        [268.2669, 105.9949]])

In [22]:
print(
    dataset[0]["observation.image"].shape
)  # (2, c, h, w) -> (2, 3, 96, 96) # observation image horizon is 2
print(
    dataset[0]["observation.state"].shape
)  # (6, c) -> (6, 2) # observation horizon is 2
print(dataset[0]["action"].shape)  # (16, c) -> (16, 2) # action horizon is 16

torch.Size([2, 3, 96, 96])

torch.Size([2, 2])

torch.Size([16, 2])

In [23]:
dataloader = torch.utils.data.DataLoader(
    dataset,
    num_workers=0,
    batch_size=64,
    shuffle=True,
)

batch = next(iter(dataloader))
print(
    f"{batch['observation.image'].shape=}"
)  # (64, 2, c, h, w) # (64, 2, 3, 96, 96) # (Batch, To, Co, H, W)
print(
    f"{batch['observation.state'].shape=}"
)  # (64, 2, c) # (64, 2, 2) #(Batch, To, Do)
print(f"{batch['action'].shape=}")  # (32, 64, c) # (32, 64, 2) # (Batch, Ta, Da)

batch['observation.image'].shape=torch.Size([64, 2, 3, 96, 96])

batch['observation.state'].shape=torch.Size([64, 2, 2])

batch['action'].shape=torch.Size([64, 16, 2])

In [24]:
batch["observation.image"]

tensor([[[[[1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000],
           [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
           [0.9686, 0.9137, 1.0000,  ..., 1.0000, 0.9137, 0.9686],
           ...,
           [0.9686, 0.9137, 1.0000,  ..., 1.0000, 0.9137, 0.9686],
           [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
           [1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000]],

          [[1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000],
           [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
           [0.9686, 0.9137, 1.0000,  ..., 1.0000, 0.9137, 0.9686],
           ...,
           [0.9686, 0.9137, 1.0000,  ..., 1.0000, 0.9137, 0.9686],
           [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
           [1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000]],

          [[1.0000, 0.9725, 0.9725,  ..., 0.9725, 0.9725, 1.0000],
           [0.9725, 0.8706, 0.9137,  ..., 0.9137, 0.8706, 0.9725],
           [0.9686, 0.9137

In [25]:
batch["observation.state"]

tensor([[[311.0941, 286.2932],
         [311.4899, 286.1429]],

        [[322.5814, 240.2250],
         [313.9416, 235.5182]],

        [[198.7455, 392.7512],
         [207.9234, 403.2974]],

        [[294.9893, 205.2175],
         [292.2089, 207.9238]],

        [[107.5618, 427.8309],
         [106.5285, 423.8342]],

        [[391.9460, 398.4852],
         [390.1347, 398.8326]],

        [[185.0149, 276.7139],
         [186.2959, 280.8198]],

        [[210.6938, 347.1907],
         [218.8033, 352.9165]],

        [[176.6409, 290.2641],
         [168.2863, 291.1133]],

        [[280.0103, 349.5991],
         [279.7092, 348.1023]],

        [[ 69.2197, 233.8038],
         [ 69.4781, 246.6330]],

        [[345.5142, 304.1718],
         [337.6278, 311.0691]],

        [[209.8434, 371.7256],
         [208.5795, 370.5362]],

        [[352.6145, 120.7170],
         [360.9249, 127.3581]],

        [[133.4246, 283.9660],
         [138.3328, 294.1839]],

        [[343.3618, 108.9553],
         

## Original Dataset


In [26]:
import os

In [27]:
import importlib.util

In [28]:
#Path(__file__).parent.parent.parent / "diffusion_policy_repo"  # If in a .py file


In [29]:
dataset_path = Path.cwd().parent.parent / "pusht" / "pusht_cchi_v7_replay.zarr"

# parameters
pred_horizon = 16
obs_horizon = 2
action_horizon = 8
#|o|o|                             observations: 2
#| |a|a|a|a|a|a|a|a|               actions executed: 8
#|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p| actions predicted: 16

# create dataset from file
dataset = PushTImageDataset(
    dataset_path=dataset_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon
)
# save training data statistics (min, max) for each dim
stats = dataset.stats

# create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=64,
    num_workers=4,
    shuffle=True,
    # accelerate cpu-gpu transfer
    pin_memory=True,
    # don't kill worker process afte each epoch
    persistent_workers=True
)

# visualize data in batch
batch = next(iter(dataloader))
print("batch['image'].shape:", batch['image'].shape)
print("batch['agent_pos'].shape:", batch['agent_pos'].shape)
print("batch['action'].shape", batch['action'].shape)

batch['image'].shape:
torch.Size([64, 2, 3, 96, 96])

batch['agent_pos'].shape:
torch.Size([64, 2, 2])

batch['action'].shape
torch.Size([64, 16, 2])

# Network: Deniosing (Noise Prediction)


In [30]:
import torch
import torch.nn as nn
import torchvision

from typing import Tuple, Sequence, Dict, Union, Optional, Callable

In [31]:
resnet_func = getattr(torchvision.models, "resnet18")
resnet_model = resnet_func(weights=None)
resnet_model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [32]:
torch.nn.Identity()

Identity()

In [33]:
# using standard ResNet implementation from torchvision
def get_resnet(name: str, weights=None, **kwargs) -> nn.Module:
    resnet_func = getattr(torchvision.models, "resnet18")
    resnet_model = resnet_func(weights=weights, **kwargs)

    # remove the final classification layer for resnet18 512 to empty
    # https://docs.pytorch.org/docs/stable/generated/torch.nn.Identity.html
    resnet_model.fc = torch.nn.Identity()
    return resnet_model


# construct a ResNet model with no pretrained weights
vision_encoder = get_resnet("resnet18", weights=None)
print(vision_encoder)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (layer2): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (downsample): Sequential(
        (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (1): BasicBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (layer3): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (downsample): Sequential(
        (0): Conv2d(128, 256, kernel_size=(1, 1), stride=(2, 2), bias=False)
        (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (1): BasicBlock(
      (conv1): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (layer4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(256, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (downsample): Sequential(
        (0): Conv2d(256, 512, kernel_size=(1, 1), stride=(2, 2), bias=False)
        (1): 

In [34]:
def replace_submodule(  # replace resnet batchnorm with groupnorm
    root_module: nn.Module,
    predicate: Callable[[nn.Module], bool],
    func: Callable[[nn.Module], nn.Module],
) -> nn.Module:
    """
    Replace all submodules in `root_module` that satisfy the `predicate` with the result of `func`.
    The `predicate` is a function that takes a module and returns True if the module should be replaced.
    The `func` is a function that takes a module and returns a new module to replace
    the original one.
    """
    if predicate(root_module):
        return func(root_module)

    bn_list = [
        k.split(".")
        for k, m in root_module.named_modules(remove_duplicate=True)
        if predicate(m)
    ]
    for *parent, k in bn_list:
        parent_module = root_module
        if len(parent) > 0:
            parent_module = root_module.get_submodule(".".join(parent))
        if isinstance(parent_module, nn.Sequential):
            src_module = parent_module[int(k)]
        else:
            src_module = getattr(parent_module, k)
        tgt_module = func(src_module)
        if isinstance(parent_module, nn.Sequential):
            parent_module[int(k)] = tgt_module
        else:
            setattr(parent_module, k, tgt_module)
    # verify that all modules are replaced
    bn_list = [
        k.split(".")
        for k, m in root_module.named_modules(remove_duplicate=True)
        if predicate(m)
    ]
    assert len(bn_list) == 0
    return root_module


def replace_bn_with_gn(
    root_module: nn.Module,  # the root resnet module of the model
    features_per_group: int = 16,  # number of features per group for GroupNorm, e.g. 16
) -> nn.Module:
    """Replace all BatchNorm2d modules with GroupNorm."""
    replace_submodule(
        root_module=root_module,
        predicate=lambda x: isinstance(x, nn.BatchNorm2d),
        func=lambda x: nn.GroupNorm(
            num_groups=x.num_features // features_per_group, num_channels=x.num_features
        ),
    )
    return root_module


vision_encoder = replace_bn_with_gn(vision_encoder)
print(vision_encoder)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): GroupNorm(4, 64, eps=1e-05, affine=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): GroupNorm(4, 64, eps=1e-05, affine=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): GroupNorm(4, 64, eps=1e-05, affine=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): GroupNorm(4, 64, eps=1e-05, affine=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): GroupNorm(4, 64, eps=1e-05, affine=True)
    )
  )
  (layer2): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn1): GroupNorm(8, 128, eps=1e-05, affine=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): GroupNorm(8, 128, eps=1e-05, affine=True)
      (downsample): Sequential(
        (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
        (1): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
    )
    (1): BasicBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): GroupNorm(8, 128, eps=1e-05, affine=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): GroupNorm(8, 128, eps=1e-05, affine=True)
    )
  )
  (layer3): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn1): GroupNorm(16, 256, eps=1e-05, affine=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): GroupNorm(16, 256, eps=1e-05, affine=True)
      (downsample): Sequential(
        (0): Conv2d(128, 256, kernel_size=(1, 1), stride=(2, 2), bias=False)
        (1): GroupNorm(16, 256, eps=1e-05, affine=True)
      )
    )
    (1): BasicBlock(
      (conv1): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): GroupNorm(16, 256, eps=1e-05, affine=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): GroupNorm(16, 256, eps=1e-05, affine=True)
    )
  )
  (layer4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(256, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn1): GroupNorm(32, 512, eps=1e-05, affine=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): GroupNorm(32, 512, eps=1e-05, affine=True)
      (downsample): Sequential(
        (0): Conv2d(256, 512, kernel_size=(1, 1), stride=(2, 2), bias=False)
        (1): GroupNorm(32, 512, eps=1e-05, affine=True)
      )
    )
    (1): BasicBlock(
      (conv1): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): GroupNorm(32, 512, eps=1e-05, affine=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): GroupNorm(32, 512, eps=1e-05, affine=True)
    )
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(1, 1))
  (fc): Identity()
)

In [35]:
# set ResNet output dim of 512
vision_features_dim = 512
# agent_pos is 2 dimensions
lowdim_obs_dim = 2
# observation feature has 514 dims in total per step
obs_dim = vision_features_dim + lowdim_obs_dim
action_dim = 2  # action is 2 dimensions (x, y)

In [36]:
obs_dim

514

# Unet architecture


## Time Position Embedding


In [37]:
import math

In [38]:
t = 1
half_dim = 512 // 2
half_dim_min1 = half_dim - 1
emb = math.log(10000) / half_dim_min1
emb = torch.exp(torch.arange(half_dim) * -emb)
t = 

SyntaxError: invalid syntax (1417869075.py, line 6)

In [39]:
torch.arange(512 // 2)

tensor([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
         14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
         28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
         42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
         56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
         70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,
         84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,
         98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
        112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
        126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139,
        140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153,
        154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167,
        168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 1

In [40]:
x = torch.randn(1, 2, 16)
t = torch.zeros((1,))  # torch.Size([1])
t.expand(x.shape[0]).shape

torch.Size([1])

In [41]:
dim = 512
half_dim = dim // 2
emb = math.log(10000) / (half_dim - 1)  # 0.036118981850886994
emb = torch.exp(torch.arange(half_dim) * -emb)  # torch.Size([256])
emb = t[:, None] * emb[None, :]  # (torch.Size([1, 1]), torch.Size([1, 256]))
emb.shape

torch.Size([1, 256])

In [42]:
class SinusoidalPosEmb(nn.Module):
    """
    dim: Dimension of time embedding 256
    """

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        """
        t: conditioning timestep (B,)
        """
        device = t.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb


class SinusoidalEncoder(nn.Module):
    """
    dsed: Dimension of time embedding (B, 256)
    """

    def __init__(self, dsed):
        super().__init__()
        self.pos_emb = SinusoidalPosEmb(dsed)

        self.mlp = nn.Sequential(
            nn.Linear(dsed, dsed * 4),
            nn.Mish(),
            nn.Linear(dsed * 4, dsed),
        )

    def forward(self, t):
        emb = self.pos_emb(t)
        return self.mlp(emb)


# SinusoidalPosEmb(256)(t) #works
SinusoidalEncoder(256)(t).shape

torch.Size([1, 256])

In [43]:
torch.zeros((1,)).shape

torch.Size([1])

In [44]:
class Conv1dBlock(nn.Module):
    """
    Single convolutional block

    Conv1d --> GroupNorm --> Mish

    #Todo:
        - Investigate GroupNorm alternatives.
        - Investigate Mish alternatives # Paper recommends Mish
        - Implement Unet-Model API Pipeline to Iterate different designs
        - Investigate padding as kernel_size // 2
        - Investigate number of groups as 8.

    #Done:
        - GroupNorm over BachNorm is still the best for CNN based Unet (LayerNorm,InstanceNorm are alternatives)
        - Mish over Relu is best non-monotonic, continuously differentiable, stability
        - padding = kernel_size // 2 : This is common practice to preserve spacial dim
        - group size 8 provides good trade-offs between channel groupinf and statistical robustness

    #Notes:
        - GroupNorm is generally preffered for CNN Unet
        - BatchSize Sensitivity and Compatibility with EMA training is the reason why BatchNorm is discouraged.
        - Similar alternatives to Mish includes Swish/SiLU but GELU is more for Transformers and ViT
    """

    def __init__(
        self, in_channels, out_channels, kernel_size, n_groups=8, activation="Mish"
    ):
        super().__init__()

        # Todo: add different activation options
        # activations = {"Mish":nn.Mish(), "Swish":nn.Swish(), "SiLU":nn.SiLU()}
        self.conv1dblock = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size // 2),
            nn.GroupNorm(n_groups, out_channels),
            nn.Mish(),  # activations[activation]
        )

    def forward(self, x):
        return self.conv1dblock(x)


x = torch.randn(1, 2, 16)
conv_output = Conv1dBlock(2, 256, 3)(x)  # torch.Size([1, 256, 16])

In [45]:
Conv1dBlock(2, 256, 3)

Conv1dBlock(
  (conv1dblock): Sequential(
    (0): Conv1d(2, 256, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): GroupNorm(8, 256, eps=1e-05, affine=True)
    (2): Mish()
  )
)

In [46]:
class Downsample1d(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Conv1d(dim, dim, 3, 2, 1)

    def forward(self, x):
        return self.conv(x)


class Upsample1d(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.ConvTranspose1d(dim, dim, 4, 2, 1)

    def forward(self, x):
        return self.conv(x)

In [47]:
class FiLMConditioning(nn.Module):
    """
    cond_dim: time embedding + global feature embedding # 250 + 514
    out_channel: block output channels
    """

    def __init__(self, cond_dim, out_channels):
        super().__init__()

        cond_channels = out_channels * 2
        self.film_encoder = nn.Sequential(
            nn.Mish(),
            nn.Linear(
                cond_dim, cond_channels
            ),  # project cond_dim to out_channels * 2 # torch.Size([B, C])
            nn.Unflatten(-1, (-1, 1)),  # (B, C, 1)
        )

    def forward(self, cond):
        return self.film_encoder(cond)


print(FiLMConditioning(770, 256)(torch.randn(1, 770)).shape)
# film_embed = FiLMConditioning(770, 256)(torch.randn(1, 770)) #torch.Size([1, 512])
# film_embed = film_embed.reshape(film_embed.shape[0], 2, 256, 1)# torch.Size([1, 2, 256, 1]) # split 512 to 2 * 256 again
# scale, bias = film_embed[:,0,...] , film_embed[:,1,...] #(torch.Size([1, 256, 1]), torch.Size([1, 256, 1]))
# print((scale * conv_output + bias).shape)
# Conv1dBlock(256,512,3)(scale * conv_output + bias).shape

torch.Size([1, 512, 1])

In [48]:
nn.Conv1d(2, 2, 1) if 2 != 2 else nn.Identity()

Identity()

In [49]:
class ConditionalResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, cond_dim, kernel_size=3, n_groups=8):
        super().__init__()

        # group conv1d block operations
        self.blocks = nn.ModuleList(
            [
                Conv1dBlock(
                    in_channels, out_channels, kernel_size, n_groups=n_groups
                ),  # in, out
                Conv1dBlock(
                    out_channels, out_channels, kernel_size, n_groups=n_groups
                ),  # out, out
            ]
        )

        self.out_channels = out_channels

        # apply film conditioning to predict per-channel scale and bias
        self.film_embed = FiLMConditioning(cond_dim, out_channels)

        # make sure dimensions are compitible
        self.residual_conv = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x, cond):
        """
        x (action_noise) : (batch_size, in_channels, horizon) -> (B, C, T)
        cond : (batch_size, cond_dim)

        #Notes:
        C : action dimensions
        cond: time_cond + obs_cond (256 + 514) -> (B, 770)

        returns:
        out : (batch_size, in_channels, horizon)
        """
        out = self.blocks[0](x)

        film_embed = self.film_embed(cond)
        film_embed = film_embed.reshape(film_embed.shape[0], 2, self.out_channels, 1)
        scale, bias = film_embed[:, 0, ...], film_embed[:, 1, ...]
        out = scale * out + bias
        #print(out.shape)

        out = self.blocks[1](out)
        out = out + self.residual_conv(x)
        return out


x = torch.randn(1, 2, 16)
cond_dim = torch.randn(1, 770)
ConditionalResidualBlock1D(2, 256, 770)(x, cond_dim).shape

torch.Size([1, 256, 16])

In [50]:
all_dims = [2] + list([256, 512, 1024])
all_dims[:-1], all_dims[1:]
in_out = list(zip(all_dims[:-1], all_dims[1:]))

In [51]:
class UnetEncoders(nn.Module):
    def __init__(self,
            in_out,
            cond_dim,
            kernel_size=5,
            n_groups=8):
        super().__init__()

        down_modules = nn.ModuleList([])
        for ind, (dim_in, dim_out) in enumerate(in_out): #(2,256),(256,512),(512,1024)
            is_last = ind >= (len(in_out) - 1)
            down_modules.append(nn.ModuleList([
                ConditionalResidualBlock1D(
                    dim_in, dim_out, cond_dim=cond_dim, kernel_size=kernel_size, n_groups=n_groups),
                ConditionalResidualBlock1D(
                    dim_out, dim_out, cond_dim=cond_dim, kernel_size=kernel_size, n_groups=n_groups),
                Downsample1d(dim_out) if not is_last else nn.Identity()
            ]))

        self.encoders = down_modules

    def __iter__(self):
        return iter(self.encoders)


down_modules = UnetEncoders(in_out, 256+514)

x = torch.randn(1, 2, 16)
cond_dim = torch.randn(1, 770)
# Access the first resnet in the first encoder block

#x = down_modules.encoders[0][0](x, cond_dim)  # first resnet
h = []
for idx, (resnet, resnet2, downsample) in enumerate(down_modules):
    x = resnet(x, cond_dim)
    x = resnet2(x, cond_dim)
    print(x.shape)
    h.append(x)
    x = downsample(x)
    print(x.shape)

torch.Size([1, 256, 16])

torch.Size([1, 256, 8])

torch.Size([1, 512, 8])

torch.Size([1, 512, 4])

torch.Size([1, 1024, 4])

torch.Size([1, 1024, 4])

In [52]:
class UnetLatent(nn.Module):
    def __init__(self,
            all_dims,
            cond_dim,
            kernel_size=5,
            n_groups=8):
        super().__init__()

        mid_dim = all_dims[-1]
        mid_modules = nn.ModuleList([
            ConditionalResidualBlock1D(
                mid_dim, mid_dim, cond_dim=cond_dim,
                kernel_size=kernel_size, n_groups=n_groups
            ),
            ConditionalResidualBlock1D(
                mid_dim, mid_dim, cond_dim=cond_dim,
                kernel_size=kernel_size, n_groups=n_groups
            ),
        ])
        self.latent = mid_modules

    def __iter__(self):
        return iter(self.latent)


x = torch.randn(1, 1024, 4)
cond_dim = torch.randn(1, 770)
all_dims = [2] + list([256, 512, 1024]) #[2, 256, 512, 1024]

resnet1, resnet2 = UnetLatent(all_dims, 770)
print(resnet1(x, cond_dim).shape)
print(resnet2(resnet1(x, cond_dim), cond_dim).shape)

torch.Size([1, 1024, 4])

torch.Size([1, 1024, 4])

In [53]:
all_dim[-1]

NameError: name 'all_dim' is not defined

In [54]:
in_out = list(zip(all_dims[:-1], all_dims[1:]))
list(reversed(in_out[1:]))

[(512, 1024), (256, 512)]

In [55]:
for i in range(len(h)):
    print(h[i].shape)

torch.Size([1, 256, 16])

torch.Size([1, 512, 8])

torch.Size([1, 1024, 4])

In [56]:
h_copy = [tensor.clone() for tensor in h]

In [57]:
x = torch.randn(1, 1024, 4)
torch.cat((x, h_copy[-1]), dim=1).shape

torch.Size([1, 2048, 4])

In [58]:
class UnetUpSample(nn.Module):
    def __init__(self,
            in_out,
            cond_dim,
            kernel_size=5,
            n_groups=8):
        super().__init__()

        up_modules = nn.ModuleList([])
        for ind, (dim_in, dim_out) in enumerate(reversed(in_out[1:])): #(512,1024),(256,512)
            is_last = ind >= (len(in_out) - 1)
            up_modules.append(nn.ModuleList([
                ConditionalResidualBlock1D(
                    dim_out*2, dim_in, cond_dim=cond_dim,
                    kernel_size=kernel_size, n_groups=n_groups),
                ConditionalResidualBlock1D(
                    dim_in, dim_in, cond_dim=cond_dim,
                    kernel_size=kernel_size, n_groups=n_groups),
                Upsample1d(dim_in) if not is_last else nn.Identity()
            ]))

        self.decoders = up_modules

    def __iter__(self):
        return iter(self.decoders)

# up_modules = UnetUpSample(in_out, 256+514)
# for idx, (resnet, resnet2, upsample) in enumerate(up_modules):
#     x = torch.cat((x, h_copy[-1]), dim=1).
#     x = resnet(x, cond_dim)
#     x = resnet2(x, cond_dim)
#     x = upsample(x)
#     print(x.shape)

up_modules = UnetUpSample(in_out, 256+514)
resnet, resnet2, upsample = up_modules.decoders[0]
x = torch.randn(1, 1024, 4)
x = torch.cat((x, h_copy[-1]), dim=1)#torch.Size([1, 2048, 4])
res1 = resnet(x, cond_dim)#torch.Size([1, 512, 4])

res2 = resnet2(res1,cond_dim) #torch.Size([1, 512, 4])

up = upsample(res2)#torch.Size([1, 512, 8])


In [59]:
torch.cat((x, h_copy[-1]), dim=1).shape

torch.Size([1, 3072, 4])

In [60]:
class ConditionalUnet1D(nn.Module):
    def __init__(self,
                input_dim,
                global_cond_dim,
                diffusion_step_embed_dim=256,
                down_dims=[256, 512, 1024],
                kernel_size=5,
                n_groups=8):

        """
            input_dim: Dim of noisy random actions for diffusion model noise prediction.
            global_cond_dim: Dim of encoded observation for FilM conditioning
            in addition to the diffusion step embedding. This is usually obs_horizon * obs_dim.
        """
        super().__init__()
        all_dims = [input_dim] + list(down_dims) #[input_dim, down_dims]
        start_dim = down_dims[0] #input_dim

        dsed = diffusion_step_embed_dim
        cond_dim = dsed + global_cond_dim #interger value

        int_out = list(zip(all_dims[:-1], all_dims[1:]))

        self.diffusion_step_encoder = SinusoidalEncoder(dsed)
        self.down_modules = UnetEncoders(in_out, cond_dim)
        self.mid_modules = UnetLatent(all_dims, cond_dim)
        self.up_modules = UnetUpSample(in_out, cond_dim)
        self.final_conv = nn.Sequential(
            Conv1dBlock(start_dim, start_dim, kernel_size=kernel_size),
            nn.Conv1d(start_dim, input_dim, 1),
        )

        print("number of parameters {:e}".format(
            sum(p.numel() for p in self.parameters())
        ))

    def forward(self,
            sample: torch.Tensor,
            timestep: Union[torch.Tensor, float, int],
            global_cond=None):

        sample = sample.moveaxis(-1,-2) #from (B,T,C)->(B,C,T)

        timesteps = timestep
        if not torch.is_tensor(timesteps): # This is essential for inference from K:int -> torch
            # TODO: this requires sync between CPU and GPU. So try to pass timesteps as tensors if you can
            timesteps = torch.tensor([timesteps], dtype=torch.long, device=sample.device)
        elif torch.is_tensor(timesteps) and len(timesteps.shape) == 0:
            timesteps = timesteps[None].to(sample.device)
        # broadcast to batch dimension in a way that's compatible with ONNX/Core ML
        timesteps = timesteps.expand(sample.shape[0])

        global_feature = self.diffusion_step_encoder(timesteps) #torch.Size([1, 256])

        if global_cond is not None:
            global_feature = torch.cat([
                global_feature, global_cond
            ], axis=-1) # concat time embed with obs embed #torch.Size([1, 1284])

        x = sample
        h = []
        # downsample
        for idx, (resnet, resnet2, downsample) in enumerate(self.down_modules):
            x = resnet(x, global_feature)
            x = resnet2(x, global_feature)
            h.append(x)
            x = downsample(x)

        # middle
        for mid_module in self.mid_modules:
            x = mid_module(x, global_feature)

        # upsample
        for idx, (resnet, resnet2, upsample) in enumerate(self.up_modules):
            x = torch.cat((x, h.pop()), dim=1)
            x = resnet(x, global_feature)
            x = resnet2(x, global_feature)
            x = upsample(x)

        #final
        x = self.final_conv(x)
        x = x.moveaxis(-1, -2) # return axis from (B,C,T)->(B,T,C)
        return x

action_dim = 2
obs_dim = 512+2 #vision_dim+action_dim ->514
global_cond_dim = obs_dim*2#obs_dim*obs_horizon -> 1028
unet = ConditionalUnet1D(action_dim, global_cond_dim)


noised_action = torch.randn(1,16,2)
obs = torch.randn(1,2,514) #resnet encoded output
flat_obs = obs.flatten(start_dim=1) #torch.Size([1, 1028])
time_step=torch.randn(1,)
denoise = unet(noised_action, time_step, flat_obs)

number of parameters 7.994727e+07

## observation Embedding


In [61]:
# Example Input: Define the observation and agent position tensors
obs_horizon = 2  # To number of past frames to consider in the observation
image = torch.zeros(
    (1, obs_horizon, 3, 96, 96)
)  # (batch_size, obs_horizon, channels, height, width) # (Batch, To, Co, H, W)
agent_pos = torch.zeros(
    (1, obs_horizon, 2)
)  # (batch_size, obs_horizon, agent_pos_dim) [agent state] # (Batch, To, Do)

In [62]:
# image.flatten(end_dim=1).shape # (batch_size * obs_horizon, channels,  height, width)
image_features = vision_encoder(
    image.flatten(end_dim=1)
)  # (batch_size * obs_horizon, 512)
image_features.shape  # ([2, 512])

torch.Size([2, 512])

In [63]:
image_features = image_features.reshape(
    *image.shape[:2], -1
)  # (batch_size, obs_horizon, 512)
image_features.shape  # (1, 2, 512)

torch.Size([1, 2, 512])

In [64]:
image_features[0]  # (2, 512) # first frame, second frame

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], grad_fn=<SelectBackward0>)

In [65]:
agent_pos[0]

tensor([[0., 0.],
        [0., 0.]])

In [66]:
obs = torch.cat(
    [image_features, agent_pos], dim=-1
)  # (batch_size, obs_horizon, 512 + 2)
obs.shape  # (1, 2, 514)

torch.Size([1, 2, 514])

## Noise prediction


In [67]:
pred_horizon = 16  # number of future actions to predict

In [68]:
action_dim  # 2  # action is 2 dimensions (x, y)

2

In [69]:
# Example Input: Define the noised action and diffusion iteration tensors
noised_action = torch.randn(
    (1, pred_horizon, action_dim)
)  # (batch_size, obs_horizon, action_dim)
diffusion_iter = torch.zeros((1,))  # (batch_size,)

In [70]:
torch.zeros((1,))

tensor([0.])

In [71]:
obs.shape

torch.Size([1, 2, 514])

In [72]:
noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,  # input_dim (action_dim)
    global_cond_dim=obs_dim * obs_horizon,  # obs_dim=(512 + 2) * obs_horizon=2 -> 1028
)

nets = nn.ModuleDict({
    'vision_encoder': vision_encoder,
    'noise_pred_net': noise_pred_net
})


print(sum(p.numel() for p in nets.parameters()))

number of parameters 7.994727e+07

91123778

In [73]:
with torch.no_grad():
    #example inputs
    image = torch.zeros((1, obs_horizon, 3, 96, 96))
    agent_pos = torch.zeros((1, obs_horizon, 2))

    # vision encoder
    image_features = nets['vision_encoder'](
        image.flatten(end_dim=1)
    )# (2,512)
    image_features = image_features.reshape(*image.shape[:2],-1)#(1,2,512)
    obs = torch.cat([image_features, agent_pos], dim=-1) #(1,2,514)

    noised_action = torch.randn((1, pred_horizon, action_dim))
    diffusion_iter = torch.zeros((1,))

    noise = nets['noise_pred_net'](
        sample=noised_action,
        timestep=diffusion_iter,
        global_cond=obs.flatten(start_dim=1)
    )

    denoised_action = noised_action - noise

# Training


In [74]:
import sys
from pathlib import Path

import wandb

In [75]:
model_path = Path.cwd().parent / "model"
sys.path.insert(0, str(model_path))

In [76]:
# At the top of your notebook
import sys
sys.path.append('../models')

from model_config import TrainingConfig
from training_setup import setup_training

In [77]:
# In training_setup.py
import importlib
import model_config
importlib.reload(model_config)
from model_config import TrainingConfig
from training_setup import setup_training

In [78]:
TrainingConfig.lr_scheduler_name

'cosine'

In [79]:
import os

In [ ]:
def train_diffusion_policy(epochs, nets, dataloader, save=False):
    """
    Main training function using the configuration structure
    """
    
    # Initialize configuration
    cfg = TrainingConfig(
        num_epochs=epochs,
        #obs_horizon=obs_horizon,
        # You can override any default values here
        # pred_horizon=your_pred_horizon,
        # action_horizon=your_action_horizon,
    )
    
    # Setup all training components
    training_components = setup_training(nets, dataloader, cfg)
    
    device = training_components['device']
    noise_scheduler = training_components['noise_scheduler']
    ema = training_components['ema']
    optimizer = training_components['optimizer']
    lr_scheduler = training_components['lr_scheduler']
    
    # Track model architecture
    #wandb.watch(nets, log="all", log_freq=100)
    
    # Training loop with enhanced logging
    with tqdm(range(cfg.num_epochs), desc='Epoch') as tglobal:
        for epoch_idx in tglobal:
            epoch_loss = []
            epoch_lr = []
            epoch_timesteps = []
            
            # Batch loop with detailed logging
            with tqdm(dataloader, desc='Batch', leave=False) as tepoch:
                for batch_idx, nbatch in enumerate(tepoch):
                    # Your existing training code...
                    nimage = nbatch['image'][:,:obs_horizon].to(device)
                    nagent_pos = nbatch['agent_pos'][:,:obs_horizon].to(device)
                    naction = nbatch['action'].to(device)
                    B = nagent_pos.shape[0]

                    # Encoder vision features
                    image_features = nets['vision_encoder'](nimage.flatten(end_dim=1))
                    image_features = image_features.reshape(*nimage.shape[:2], -1)
                    
                    # Concatenate vision feature and low-dim obs
                    obs_features = torch.cat([image_features, nagent_pos], dim=-1)
                    obs_cond = obs_features.flatten(start_dim=1)
                    
                    # Sample noise and timesteps
                    noise = torch.randn(naction.shape, device=device)
                    timesteps = torch.randint(
                        0, noise_scheduler.config.num_train_timesteps,
                        (B,), device=device
                    ).long()
                    
                    # Forward diffusion process
                    noisy_actions = noise_scheduler.add_noise(naction, noise, timesteps)
                    
                    # Predict noise
                    noise_pred = nets['noise_pred_net'](
                        noisy_actions, timesteps, global_cond=obs_cond)
                    
                    # Compute loss
                    loss = nn.functional.mse_loss(noise_pred, noise)
                    
                    # Optimization
                    loss.backward()
                    
                    # Track gradient norms (useful for debugging)
                    grad_norm = torch.nn.utils.clip_grad_norm_(nets.parameters(), max_norm=float('inf'))
                    
                    optimizer.step()
                    optimizer.zero_grad()
                    lr_scheduler.step()
                    ema.step(nets.parameters())
                    
                    # Collect metrics
                    loss_cpu = loss.item()
                    current_lr = lr_scheduler.get_last_lr()[0]
                    mean_timestep = timesteps.float().mean().item()
                    
                    epoch_loss.append(loss_cpu)
                    epoch_lr.append(current_lr)
                    epoch_timesteps.append(mean_timestep)
                    
                    # Log every N steps (avoid overwhelming wandb)
                    global_step = epoch_idx * len(dataloader) + batch_idx
                    if batch_idx % cfg.log_every_n_batches == 0:
                        wandb.log({
                            "train/batch_loss": loss_cpu,
                            "train/learning_rate": current_lr,
                            "train/grad_norm": grad_norm.item(),
                            "train/mean_timestep": mean_timestep,
                            "train/epoch": epoch_idx,
                            "train/global_step": global_step,
                        })
                    
                    # Update progress bars
                    tepoch.set_postfix(loss=loss_cpu, lr=f"{current_lr:.2e}")
                    tglobal.set_postfix(loss=np.mean(epoch_loss))
            
            # End of epoch logging
            epoch_metrics = {
                "epoch/avg_loss": np.mean(epoch_loss),
                "epoch/min_loss": np.min(epoch_loss),
                "epoch/max_loss": np.max(epoch_loss),
                "epoch/std_loss": np.std(epoch_loss),
                "epoch/final_lr": epoch_lr[-1],
                "epoch/avg_timestep": np.mean(epoch_timesteps),
                "epoch/number": epoch_idx,
            }
            
            wandb.log(epoch_metrics)
            
            #print(f"Epoch {epoch_idx}: Avg Loss = {epoch_metrics['epoch/avg_loss']:.6f}")

    # Final model logging
    wandb.log({
        "training/total_epochs": cfg.num_epochs,
        "training/final_loss": epoch_loss[-1],
    })

    # Save EMA model
    ema_nets = nets
    ema.copy_to(ema_nets.parameters())

    # Optional: Save model artifacts
    if save:
        checkpoint_name = f"{cfg.num_epochs}_model_checkpoint.pth"
        torch.save(nets.state_dict(), checkpoint_name)

        # Use the same filename for wandb
        wandb.save(checkpoint_name)
        artifact = wandb.Artifact("diffusion_policy_model", type="model")
        artifact.add_file(checkpoint_name)
        wandb.log_artifact(artifact)

    wandb.finish()    
    os.environ["WANDB_MODE"] = "disabled"  # Disable W&B for subsequent operations
    print("Training completed and logged to wandb!")
    
    return ema_nets


# Usage example:
ema_nets = train_diffusion_policy(5, nets, dataloader, False)

wandb: Currently logged in as: tegbe to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch: 100%|██████████| 5/5 [04:17<00:00, 51.54s/it, loss=0.0454]


epoch/avg_loss,█▂▂▁▁
epoch/avg_timestep,▇▂▃▁█
epoch/final_lr,▇█▅▂▁
epoch/max_loss,█▁▁▁▁
epoch/min_loss,█▅▃▂▁
epoch/number,▁▃▅▆█
epoch/std_loss,█▁▁▁▁
train/batch_loss,█▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch,▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▅▅▅▅▅▅▅▆▆▆▆▆▆▆██████████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▇▇▇▇▇▇███
train/grad_norm,█▅▄▄▃▂▂▃▂▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁


Training completed and logged to wandb!

Training completed and logged to wandb!

In [81]:
from image_dataset import normalize_data
from skvideo.io import vwrite

# Inference


In [82]:
# inference.py
import torch
import numpy as np
import collections
from tqdm import tqdm
from IPython.display import Video
from skvideo.io import vwrite
from IPython.display import Video


In [87]:
def run_inference(
    ema_nets,
    max_steps: int = 200,
    seed: int = 100000,
    save_video: bool = True,
    video_path: str = 'vis.mp4'
):
    """
    Run inference using the trained diffusion policy (no W&B logging).
    """
    # No wandb.init() here

    # Setup environment
    env.seed(seed)
    obs, info = env.reset()

    cfg = TrainingConfig()

    # Instead of setup_training(), manually build required components
    device = torch.device(cfg.device)
    noise_scheduler = DDPMScheduler(  # or whatever scheduler you use
        num_train_timesteps=cfg.num_diffusion_iters
    )

    obs_horizon = cfg.obs_horizon
    pred_horizon = cfg.pred_horizon
    action_horizon = cfg.action_horizon

    # Infer action_dim from env
    if hasattr(env.action_space, 'shape'):
        action_dim = env.action_space.shape[0]
    else:
        action_dim = 2  # fallback

    # Keep recent observations
    obs_deque = collections.deque([obs] * obs_horizon, maxlen=obs_horizon)

    imgs = [env.render(mode='rgb_array')] if save_video else []
    rewards = []
    done = False
    step_idx = 0

    with tqdm(total=max_steps, desc="Eval PushTImageEnv") as pbar:
        while not done:
            B = 1
            images = np.stack([x['image'] for x in obs_deque])
            agent_poses = np.stack([x['agent_pos'] for x in obs_deque])

            nagent_poses = normalize_data(agent_poses, stats=stats['agent_pos'])
            nimages = images

            nimages = torch.from_numpy(nimages).to(device, dtype=torch.float32)
            nagent_poses = torch.from_numpy(nagent_poses).to(device, dtype=torch.float32)

            with torch.no_grad():
                image_features = ema_nets['vision_encoder'](nimages)
                obs_features = torch.cat([image_features, nagent_poses], dim=-1)
                obs_cond = obs_features.unsqueeze(0).flatten(start_dim=1)

                noisy_action = torch.randn((B, pred_horizon, action_dim), device=device)
                naction = noisy_action

                noise_scheduler.set_timesteps(cfg.num_diffusion_iters)
                for k in noise_scheduler.timesteps:
                    noise_pred = ema_nets['noise_pred_net'](
                        sample=naction,
                        timestep=k,
                        global_cond=obs_cond
                    )
                    naction = noise_scheduler.step(
                        model_output=noise_pred,
                        timestep=k,
                        sample=naction
                    ).prev_sample

                naction = naction.detach().to('cpu').numpy()[0]
                action_pred = unnormalize_data(naction, stats=stats['action'])

                start = obs_horizon - 1
                end = start + action_horizon
                action = action_pred[start:end, :]

            for i in range(len(action)):
                obs, reward, done, _, info = env.step(action[i])
                obs_deque.append(obs)
                rewards.append(reward)
                if save_video:
                    imgs.append(env.render(mode='rgb_array'))
                step_idx += 1
                pbar.update(1)
                pbar.set_postfix(reward=reward)
                if step_idx > max_steps:
                    done = True
                    break

            if done:
                break

    score = max(rewards) if rewards else 0
    print(f'Score: {score}')

    if save_video and imgs:
        try:
            vwrite(video_path, imgs)
            Video(video_path, embed=True, width=256, height=256)
            print(f"Video saved to: {video_path}")
        except Exception as e:
            print(f"Error saving video: {e}")


# local model recall


In [88]:
run_inference(ema_nets)

Eval PushTImageEnv: 201it [00:15, 13.14it/s, reward=0.882]                          


Score: 0.9070543071128901

Video saved to: vis.mp4

In [89]:
# import wandb

# wandb.init(project="diffusion-policy-pusht", job_type="cleanup", reinit=True)
# wandb.unwatch(noise_pred_net)
# wandb.finish()

# reload_nets = nn.ModuleDict({
#     'vision_encoder': vision_encoder,
#     'noise_pred_net': noise_pred_net
# })
# reload_nets.load_state_dict(torch.load("50_model_checkpoint.pth"))

# reload_nets

# wandb model recall


In [90]:
# import wandb
# import torch

# wandb.init(project="diffusion-policy-pusht", job_type="inference", reinit=True)
# api = wandb.Api()
# run = api.run("tegbe/diffusion-policy-pusht/cfs3y9q4")

# # Download the model artifact (assuming it's named 'diffusion_policy_model' or similar)
# artifact = run.use_artifact('diffusion_policy_model:model')
# artifact_dir = artifact.download()

# # Load your model checkpoint file
# checkpoint_path = f"{artifact_dir}/model_checkpoint.pth"
# reload_model = reload_nets = nn.ModuleDict({
#     'vision_encoder': vision_encoder,
#     'noise_pred_net': noise_pred_net
# })  # instantiate your model architecture here
# reload_model.load_state_dict(torch.load(checkpoint_path))
# reload_model.eval()

# wandb.finish()

# # Now you can run inference with `model`


In [91]:
# visualize
Video('vis.mp4', embed=True, width=256, height=256)